# Unsloth SLM Evaluation for Trip Planning (True 1-by-1 Latency)
This notebook evaluates a specialized 7B trip-planning model using Unsloth.
It measures **true, real-world latency** by running:
1. **Pass@1** — 1 prompt → 1 generation (greedy, `do_sample=False`)
2. **Pass@5** — 1 prompt → 5 generations simultaneously (`temperature=0.6`, `num_return_sequences=5`)

No batch padding convoy effect. Direct, thesis-quality latency measurement.

In [1]:
!pip install uv
!uv pip install --system trl>=0.15.0 vllm ortools 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 222.2 MB/s  0:00:00
Using Python 3.12.3 environment at: /usr
error: The interpreter at /usr is externally managed, and indicates the following:

  To install Python packages system-wide, try apt install
  python3-xyz, where xyz is the package you are trying to
  install.

  If you wish to install a non-Debian-packaged Python package,
  create a virtual environment using python3 -m venv path/to/venv.
  Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
  sure you have python3-full installed.

  If you wish to install a non-Debian packaged Python application,
  it may be easiest to use pipx install xyz, which will manage a
  virtual environment for you. Make sure you have pipx installed.

  See /usr/share/doc/python3.12/README.venv for more information.

hint: Virtual environments were not considered due to the `--system` flag


In [2]:
!pip install ortools
!pip install matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 176.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ortools]m1/2 [ortools]


In [3]:
from huggingface_hub import login
import getpass
import os

print("=== Hugging Face Login ===")
hf_token = getpass.getpass("Enter your Hugging Face WRITE Token: ")
login(token=hf_token)

print("\nAuthentication successful! Ready for training.")

=== Hugging Face Login ===


Enter your Hugging Face WRITE Token:  ········



Authentication successful! Ready for training.


In [9]:
import json
import gc
import time
import re
import os
import sys
import tempfile
import subprocess
import torch
from tqdm.notebook import tqdm
from unsloth import FastLanguageModel

# ==========================================
# 1. Configuration & Initialization
# ==========================================
MODEL_NAME = "NickolasLow1/grpo_train"
SUBFOLDER = "checkpoint-240"
MAX_SEQ_LENGTH = 4096
N_SAMPLES = 5  # For Pass@5

DATA_PATH = "trip_planning_test.json"
OUTPUT_DIR = "data"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "qwen2.5_results.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [5]:
# ==========================================
# 2. Helper Functions (Execution & Verification)
# ==========================================
def extract_code(text):
    pattern = r"```python\n(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None

def extract_json_from_output(output_str):
    try:
        start_idx = output_str.find('[')
        end_idx = output_str.rfind(']')
        start_dict = output_str.find('{')
        end_dict = output_str.rfind('}')
        
        if start_idx != -1 and end_idx != -1 and (start_dict == -1 or start_idx < start_dict):
            return json.loads(output_str[start_idx:end_idx+1])
        elif start_dict != -1 and end_dict != -1:
            return json.loads(output_str[start_dict:end_dict+1])
    except json.JSONDecodeError:
        pass
    return None

def test_code_execution(code_string):
    if not code_string:
        return False, "Failed to extract code.", None
    
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as temp_file:
        temp_file.write(code_string)
        temp_file_path = temp_file.name

    try:
        result = subprocess.run([sys.executable, temp_file_path], capture_output=True, text=True, timeout=5)
        if result.returncode != 0:
            return False, f"Execution Error: {result.stderr.strip()[-100:]}", None
        
        output = result.stdout.strip()
        if "No solution" in output:
             return True, "Valid Execution (No solution feasible)", "No solution found."
             
        parsed_json = extract_json_from_output(output)
        if parsed_json is not None:
            return True, "Valid Execution (JSON output)", parsed_json
        else:
            return False, "Executed, but output was not valid JSON.", None
            
    except subprocess.TimeoutExpired:
        return False, "Execution Timeout (>5 seconds).", None
    finally:
        if os.path.exists(temp_file_path):
            os.remove(temp_file_path)

def get_ground_truth(cities_str, durations_str):
    if not cities_str or not durations_str:
        return []
    cities = cities_str.split("**")
    durations = [int(d) for d in durations_str.split("**")]
    
    gt = []
    current_start = 1
    for c, d in zip(cities, durations):
        end = current_start + d - 1
        gt.append({
            "city": c.strip().lower(),
            "start": current_start,
            "end": end
        })
        current_start = end
    return gt

def safe_int(x):
    try:
        return int(x)
    except:
        return None

def normalize(plan):
    cleaned = []
    if isinstance(plan, dict) and all(k in plan for k in ["order", "start_times", "end_times"]):
        for city in plan["order"]:
            s = safe_int(plan["start_times"].get(city))
            e = safe_int(plan["end_times"].get(city))
            if s is not None and e is not None:
                cleaned.append({"city": str(city).strip().lower(), "start": s, "end": e})
        return sorted(cleaned, key=lambda x: x["start"])

    if isinstance(plan, dict):
        for key in ["solutions", "trip_plan", "plan", "schedule"]:
            if key in plan:
                return normalize(plan[key])

    if isinstance(plan, dict):
        for city, dates in plan.items():
            if isinstance(dates, list) and len(dates) >= 2:
                s = safe_int(dates[0])
                e = safe_int(dates[1])
                if s is not None and e is not None:
                    cleaned.append({"city": str(city).strip().lower(), "start": s, "end": e})
        if cleaned:
            return sorted(cleaned, key=lambda x: x["start"])

    if isinstance(plan, list):
        for x in plan:
            if not isinstance(x, dict):
                continue
            city_val = x.get("city") or x.get("City") or x.get("city_name")
            if not city_val:
                continue
            s = safe_int(x.get("start") or x.get("start_day") or x.get("arrival"))
            e = safe_int(x.get("end") or x.get("end_day") or x.get("departure"))
            if s is not None and e is not None:
                cleaned.append({"city": str(city_val).strip().lower(), "start": s, "end": e})
    return sorted(cleaned, key=lambda x: x["start"])

def verify_plan(generated_plan, cities_str, durations_str):
    if generated_plan is None or generated_plan == "No solution found.":
        return False
    gt_answer = normalize(get_ground_truth(cities_str, durations_str))
    try:
        norm_sol = normalize(generated_plan)
        if norm_sol == gt_answer:
            return True
    except Exception:
        pass
    return False

In [11]:
from huggingface_hub import snapshot_download

# ==========================================
# 3. Load Model and Data
# ==========================================
with open(DATA_PATH, "r", encoding="utf-8") as f:
    val_data_raw = json.load(f)

val_data_list = list(val_data_raw.values()) if isinstance(val_data_raw, dict) else val_data_raw
eval_dataset = []
keys = list(val_data_raw.keys()) if isinstance(val_data_raw, dict) else [str(i) for i in range(len(val_data_raw))]

for item in val_data_list:
    eval_dataset.append({
        "prompt": item.get("prompt_0shot") or item.get("prompt"),
        "cities": item.get("cities", ""),
        "durations": item.get("durations", "")
    })
print(f"Loaded {len(eval_dataset)} prompts.")

print("Downloading checkpoint from Hugging Face...")
# Download the adapter locally to avoid the 'subfolder' API bug
local_dir = snapshot_download(
    repo_id = MODEL_NAME, 
    allow_patterns = f"{SUBFOLDER}/*", 
    token = True
)
adapter_path = os.path.join(local_dir, SUBFOLDER)

print(f"Loading Unsloth Model from {adapter_path}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = adapter_path, # Load from the local path
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    fast_inference = True,
)
FastLanguageModel.for_inference(model)

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Model ready.")



Loaded 160 prompts.
Loading Unsloth Model from /workspace/work/hf-cache/models--NickolasLow1--grpo_train/snapshots/fb2e08309f94b0a8692b190406efe39fc47885ef/checkpoint-240...
INFO 05-05 09:40:57 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.16.1.dev0+g89a77b108.d20260417.cu128.
   \\   /|    NVIDIA RTX 6000 Ada Generation. Num GPUs = 1. Max memory: 47.383 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Standby mode is enabled. Changing `gpu_memory_utilization` to 0.87875.
Unsloth: vLLM loading unsloth/Qwen2.5-Coder-7b-Instruct-bnb-4bit with actual GPU utilization = 86.86%
Unsloth: Your GPU has CUDA compute capability 8

/opt/venv/lib/python3.12/site-packages/pydantic/type_adapter.py:607: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 05-05 09:41:16 [model.py:529] Resolved architecture: Qwen2ForCausalLM
INFO 05-05 09:41:16 [model.py:1549] Using max model len 4096
INFO 05-05 09:41:16 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=8192.
Unsloth: vLLM Bitsandbytes config using kwargs = {'load_in_8bit': False, 'load_in_4bit': True, 'bnb_4bit_compute_dtype': 'bfloat16', 'bnb_4bit_quant_storage': 'uint8', 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': [], 'llm_int8_threshold': 6.0}
INFO 05-05 09:41:16 [vllm.py:689] Asynchronous scheduling is enabled.
INFO 05-05 09:41:22 [core.py:97] Initializing a V1 LLM engine (v0.16.1.dev0+g89a77b108.d20260417) with config: model='unsloth/Qwen2.5-Coder-7b-Instruct-bnb-4bit', speculative_config=None, tokenizer='unsloth/Qwen2.5-Coder-7b-Instruct-bnb-4bit', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=N

/opt/venv/lib/python3.12/site-packages/pydantic/type_adapter.py:607: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 05-05 09:41:23 [topk_topp_sampler.py:47] Using FlashInfer for top-p & top-k sampling.
INFO 05-05 09:41:23 [gpu_model_runner.py:4124] Starting to load model unsloth/Qwen2.5-Coder-7b-Instruct-bnb-4bit...
INFO 05-05 09:41:23 [cuda.py:367] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].
INFO 05-05 09:41:23 [bitsandbytes_loader.py:786] Loading weights with BitsAndBytes quantization. May take a while ...


...

INFO 05-05 09:43:32 [core.py:278] init engine (profile, create kv cache, warmup model) took 113.50 seconds
INFO 05-05 09:43:34 [llm.py:355] Supported tasks: ('generate',)
Unsloth: Just some info: will skip parsing ['q_norm', 'ffn_norm', 'layer_norm2', 'norm1', 'norm2', 'post_feedforward_layernorm', 'layer_norm1', 'post_layernorm', 'norm', 'input_layernorm', 'pre_feedforward_layernorm', 'k_norm', 'attention_norm', 'post_attention_layernorm']


Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in Qwen2ForCausalLM is bfloat16. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", dtype=torch.float16)`
Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in Qwen2Model is bfloat16. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", dtype=torch.float16)`


Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['q_norm', 'ffn_norm', 'layer_norm2', 'norm1', 'norm2', 'post_feedforward_layernorm', 'layer_norm1', 'post_layernorm', 'norm', 'input_layernorm', 'cross_attn_input_layernorm', 'pre_feedforward_layernorm', 'k_norm', 'cross_attn_post_attention_layernorm', 'attention_norm', 'post_attention_layernorm']


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model ready.


In [13]:
import os

# ==========================================
# 4. Evaluation Loop (1-by-1 True Latency)
# ==========================================
pass1_success = 0
pass5_success = 0
total_latency_pass1 = 0.0
total_latency_pass5 = 0.0
total_out_tokens_pass1 = 0
total_out_tokens_pass5 = 0
detailed_logs = []
processed_ids = set()

# --- RESUMPTION LOGIC ---
if os.path.exists(OUTPUT_FILE):
    print(f"Found existing save file at {OUTPUT_FILE}. Resuming...")
    try:
        with open(OUTPUT_FILE, "r") as f:
            saved_data = json.load(f)
            if "detailed_logs" in saved_data:
                detailed_logs = saved_data["detailed_logs"]
                
                # Restore all counters from the saved logs
                for log in detailed_logs:
                    processed_ids.add(log["id"])
                    
                    if log.get("pass_1_success"): pass1_success += 1
                    if log.get("pass_5_success"): pass5_success += 1
                        
                    pass_1_log = log.get("pass_1_log", {})
                    total_latency_pass1 += pass_1_log.get("latency_seconds", 0.0)
                    total_out_tokens_pass1 += pass_1_log.get("output_tokens", 0)
                    total_latency_pass5 += log.get("pass_5_latency_seconds", 0.0)
                    
                    for attempt in log.get("pass_5_attempts", []):
                        total_out_tokens_pass5 += attempt.get("output_tokens", 0)
                        
        print(f"Resumed {len(processed_ids)} previously evaluated prompts.")
    except Exception as e:
        print(f"Error loading saved file, starting fresh: {e}")
# ------------------------

for k_id, item in tqdm(zip(keys, eval_dataset), total=len(eval_dataset), desc="Evaluating Prompts 1-by-1"):
    # Skip prompts we already evaluated before the crash
    if k_id in processed_ids:
        continue 

    p_str = item["prompt"]
    c_str = item["cities"]
    d_str = item["durations"]
    gt_json = get_ground_truth(c_str, d_str)

    # Format single prompt
    formatted_prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": p_str}],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to("cuda")
    prompt_length = inputs.input_ids.shape[1]

    # ----------------------------------------
    # PASS@1: True 1-by-1 Latency (Greedy, Temp = 0.01)
    # ----------------------------------------
    start_time_1 = time.time()
    with torch.inference_mode():
        outputs_1 = model.generate(
            **inputs,
            max_new_tokens=3072,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True
        )
    latency_1 = time.time() - start_time_1
    total_latency_pass1 += latency_1

    new_tokens_1 = outputs_1[:, prompt_length:]
    tok_1 = (new_tokens_1 != tokenizer.pad_token_id).sum().item()
    total_out_tokens_pass1 += tok_1
    text_1 = tokenizer.decode(new_tokens_1[0], skip_special_tokens=True)

    # Verify Pass@1
    code_1 = extract_code(text_1)
    is_syn_1, msg_1, gen_json_1 = test_code_execution(code_1)
    is_func_1 = False
    if is_syn_1 and verify_plan(gen_json_1, c_str, d_str):
        is_func_1 = True
        pass1_success += 1

    pass1_log = {
        "generated_text": text_1,
        "extracted_code": code_1,
        "execution_message": msg_1,
        "parsed_json": gen_json_1,
        "syntax_passed": is_syn_1,
        "functional_passed": is_func_1,
        "output_tokens": tok_1,
        "latency_seconds": round(latency_1, 4)
    }

    # ----------------------------------------
    # PASS@5: True Single-User Latency (Sampling, Temp = 0.6)
    # ----------------------------------------
    start_time_5 = time.time()
    with torch.inference_mode():
        outputs_5 = model.generate(
            **inputs,
            max_new_tokens=3072,
            temperature=0.6,
            do_sample=True,
            num_return_sequences=N_SAMPLES,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True
        )
    latency_5 = time.time() - start_time_5
    total_latency_pass5 += latency_5

    new_tokens_5 = outputs_5[:, prompt_length:]
    toks_5_list = (new_tokens_5 != tokenizer.pad_token_id).sum(dim=-1).tolist()
    total_out_tokens_pass5 += sum(toks_5_list)
    texts_5 = tokenizer.batch_decode(new_tokens_5, skip_special_tokens=True)

    # Verify Pass@5
    attempt_logs_5 = []
    prompt_pass5 = False
    for idx, (text_5, tok_count_5) in enumerate(zip(texts_5, toks_5_list)):
        code_5 = extract_code(text_5)
        is_syn_5, msg_5, gen_json_5 = test_code_execution(code_5)
        is_func_5 = False
        if is_syn_5 and verify_plan(gen_json_5, c_str, d_str):
            is_func_5 = True
            prompt_pass5 = True

        attempt_logs_5.append({
            "attempt_index": idx + 1,
            "generated_text": text_5,
            "extracted_code": code_5,
            "execution_message": msg_5,
            "parsed_json": gen_json_5,
            "syntax_passed": is_syn_5,
            "functional_passed": is_func_5,
            "output_tokens": tok_count_5
        })

    if prompt_pass5:
        pass5_success += 1

    detailed_logs.append({
        "id": k_id,
        "prompt": p_str,
        "ground_truth": gt_json,
        "pass_1_success": is_func_1,
        "pass_5_success": prompt_pass5,
        "pass_1_log": pass1_log,
        "pass_5_latency_seconds": round(latency_5, 4),
        "pass_5_attempts": attempt_logs_5
    })
    
    # ----------------------------------------
    # AUTO-SAVE CHECKPOINTING (Every 2 Prompts)
    # ----------------------------------------
    if len(detailed_logs) % 2 == 0 or len(detailed_logs) == len(eval_dataset):
        num_processed = len(detailed_logs)
        t_summary = {
            "Total_Pass@1_Accuracy_%": round((pass1_success / num_processed * 100), 2) if num_processed > 0 else 0,
            "Total_Pass@5_Accuracy_%": round((pass5_success / num_processed * 100), 2) if num_processed > 0 else 0,
            "Avg_Latency_Pass1_s": round((total_latency_pass1 / num_processed), 4) if num_processed > 0 else 0,
            "Avg_Latency_Pass5_s": round((total_latency_pass5 / num_processed), 4) if num_processed > 0 else 0,
            "Avg_Output_Tokens_Pass1": round((total_out_tokens_pass1 / num_processed), 2) if num_processed > 0 else 0,
            "Avg_Output_Tokens_per_attempt_Pass5": round((total_out_tokens_pass5 / (num_processed * N_SAMPLES)), 2) if num_processed > 0 else 0,
            "Total_Prompts": num_processed
        }
        with open(OUTPUT_FILE, "w") as f:
            json.dump({"summary": t_summary, "detailed_logs": detailed_logs}, f, indent=4)

# ==========================================
# 5. Record and Print Final Summary
# ==========================================
print("\n" + "="*40)
print("  TRUE LATENCY EVALUATION SUMMARY")
print("="*40)
print(json.dumps(t_summary, indent=4))
print(f"\nSaved detailed log to: {OUTPUT_FILE}")


Evaluating Prompts 1-by-1:   0%|          | 0/160 [00:00<?, ?it/s]


  TRUE LATENCY EVALUATION SUMMARY
{
    "Total_Pass@1_Accuracy_%": 79.38,
    "Total_Pass@5_Accuracy_%": 86.88,
    "Avg_Latency_Pass1_s": 49.0248,
    "Avg_Latency_Pass5_s": 71.4554,
    "Avg_Output_Tokens_Pass1": 1494.28,
    "Avg_Output_Tokens_per_attempt_Pass5": 1501.26,
    "Total_Prompts": 160
}

Saved detailed log to: data/qwen2.5_results.json
